# 05 - Neuro-Symbolic Entity Linking & Text2SQL (Batch Processing)

This notebook demonstrates **neuro-symbolic entity linking** and **Text2SQL**
patterns applied to Uruguay's crime-statistics datasets, drawing on the
framework described in [arXiv:2508.13678](https://arxiv.org/abs/2508.13678).

**Model strategy** (A100 GPU):
- **Qwen2.5-7B-Instruct**: Fast Spanish NLP, entity linking, Text2SQL
- **QwQ-32B**: Deep reasoning tasks (NSVIF semantic evaluation, causal analysis)

**Critical design principle**: All LLM inference uses **batch processing** to
maximize GPU utilization. We never process items one-by-one — instead we
compose full batches (all columns at once, all queries at once, all constraints
at once) and send them in a single inference call or minimal batched calls.

### What this notebook covers

1. **Entity Linking** — Match columns across ALL 6 datasets in a single batch prompt
2. **Text2SQL** — Batch multiple Spanish queries into one inference call
3. **Semantic Constraint Evaluation** — Evaluate ALL constraints for a dataset in one pass

In [1]:
!pip install -q transformers torch accelerate polars duckdb bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00


In [2]:
import json
from pathlib import Path

import duckdb
import polars as pl
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# ---------------------------------------------------------------------------
# Load Models — two-model strategy for A100
# ---------------------------------------------------------------------------
# Qwen2.5-7B for fast entity linking & Text2SQL (high throughput)
# QwQ-32B for deep reasoning / NSVIF semantic evaluation
#
# We load Qwen first (smaller), use it for batch tasks, then swap to QwQ
# for the reasoning-heavy task. This avoids VRAM fragmentation.

FAST_MODEL = "Qwen/Qwen2.5-7B-Instruct"     # entity linking, Text2SQL
REASONING_MODEL = "Qwen/QwQ-32B"              # NSVIF semantic evaluation


def load_model(model_id: str):
    """Load a model with optimal A100 settings."""
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,   # A100 native dtype for max throughput
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"  Loaded on {next(model.parameters()).device}")
    return tokenizer, model


def unload_model(model, tokenizer):
    """Free GPU memory before loading next model."""
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("  GPU memory freed.")


def batch_inference(tokenizer, model, prompts: list[str], max_new_tokens: int = 1024) -> list[str]:
    """Run batched inference over multiple prompts in a SINGLE forward pass.

    This is the critical optimization: instead of calling the model N times
    (one per prompt), we batch all prompts together and process them in one
    GPU pass. This maximizes GPU utilization and avoids the overhead of
    repeated model calls.
    """
    # Apply chat template to each prompt
    texts = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        texts.append(text)

    # Tokenize as a batch with padding
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    # Single batched forward pass
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    # Decode each output, skipping input tokens
    results = []
    for i, ids in enumerate(output_ids):
        input_len = inputs["input_ids"][i].ne(tokenizer.pad_token_id or 0).sum()
        generated = ids[input_len:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        results.append(text)

    return results


def single_inference(tokenizer, model, prompt: str, max_new_tokens: int = 2048) -> str:
    """Single prompt inference — used only when the prompt itself is a mega-batch
    (i.e., all work is packed into one prompt to avoid multiple calls)."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Load fast model first
tokenizer, model = load_model(FAST_MODEL)

Loading Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

  Loaded on cuda:0


In [4]:
# ---------------------------------------------------------------------------
# Download from GCS + BATCH LOAD — All 6 datasets at once
# ---------------------------------------------------------------------------
from google.colab import auth
auth.authenticate_user()

GCS_BUCKET = "gs://interior-minister-data-lake-fabled-imagery-488015-p6/processed/tabular"
DATA_DIR = Path("/content/data/tabular")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Map logical names to actual GCS filenames
GCS_FILES = {
    "delitos_denunciados": "delitos_denuncias.parquet",
    "violencia_domestica": "violencia_domestica.parquet",
    "delitos_sexuales": "delitos_sexuales.parquet",
    "homicidios_mujeres": "homicidios_mujeres.parquet",
    "medidas_alternativas": "medidas_alternativas.parquet",
    "sistema_carcelario": "sistema_carcelario.parquet",
}

# Synthetic fallback schemas (used if parquet files don't exist yet)
SYNTHETIC_SCHEMAS = {
    "delitos_denunciados": ["anio", "mes", "trimestre", "departamento", "seccional", "titulo", "subtitulo", "fecha", "sexo", "cantidad"],
    "violencia_domestica": ["ano", "departamento_hecho", "titulo", "jurisdiccion", "sexo_victima", "sexo_agresor", "fecha_denuncia", "tipo_violencia", "total"],
    "delitos_sexuales": ["anio", "departamento", "jurisdiccion", "titulo", "sexo_victima", "fecha_denuncia", "total"],
    "homicidios_mujeres": ["anio", "departamento", "vinculo", "fecha", "tipo_homicidio", "total"],
    "medidas_alternativas": ["anio", "mes", "departamento", "tipo_medida", "genero", "medio_control", "cantidad"],
    "sistema_carcelario": ["anio", "mes", "establecimiento", "sexo", "grupo_edad", "delito", "nacionalidad", "cantidad"],
}

for name, gcs_name in GCS_FILES.items():
    !gsutil -q cp {GCS_BUCKET}/{gcs_name} {DATA_DIR}/{gcs_name}

all_columns = {}
dataframes = {}

for name, gcs_name in GCS_FILES.items():
    path = DATA_DIR / gcs_name
    if path.exists():
        df = pl.read_parquet(path)
        all_columns[name] = df.columns
        dataframes[name] = df
        print(f"  {name}: {df.shape} loaded")
    else:
        all_columns[name] = SYNTHETIC_SCHEMAS[name]
        print(f"  {name}: using synthetic schema ({len(SYNTHETIC_SCHEMAS[name])} cols)")

print(f"\nTotal datasets: {len(all_columns)}")

  delitos_denunciados: (2423286, 15) loaded
  violencia_domestica: (532172, 19) loaded
  delitos_sexuales: (93438, 9) loaded
  homicidios_mujeres: (834, 15) loaded
  medidas_alternativas: (634, 19) loaded
  sistema_carcelario: (5012, 21) loaded

Total datasets: 6


In [5]:
# ---------------------------------------------------------------------------
# BATCH Entity Linking — ALL 6 datasets in ONE inference call
# ---------------------------------------------------------------------------
# Instead of comparing datasets pairwise (15 pairs = 15 LLM calls),
# we send ALL column schemas in a SINGLE prompt and ask the model to
# identify ALL cross-dataset matches at once.
#
# This is the batch-processing lesson from project-1: never item-by-item.

schema_block = "\n".join(
    f"Dataset '{name}': {json.dumps(cols)}"
    for name, cols in all_columns.items()
)

entity_linking_prompt = (
    "You are a data integration expert working with Uruguayan crime statistics.\n"
    "\n"
    "Here are the column schemas for ALL 6 datasets from Uruguay's Interior Ministry:\n\n"
    + schema_block +
    "\n\n"
    "Task: Identify ALL semantically equivalent columns ACROSS datasets in one pass.\n"
    "Consider:\n"
    "- Different spellings of the same concept (anio vs ano vs year)\n"
    "- Prefixed/suffixed variants (departamento vs departamento_hecho)\n"
    "- Same concept different names (sexo vs genero vs sexo_victima)\n"
    "- Temporal columns (fecha, fecha_denuncia, anio+mes)\n"
    "- Count columns (cantidad vs total)\n"
    "\n"
    "Return a JSON list of match groups. Each group has:\n"
    '  "concept": canonical name for this dimension\n'
    '  "matches": list of {"dataset": str, "column": str}\n'
    '  "join_type": "exact" | "fuzzy" | "derived"\n'
    '  "notes": any caveats\n'
    "\n"
    "Include ALL meaningful matches. This is for building a unified data model."
)

print(f"Sending ALL 6 dataset schemas in ONE prompt ({len(entity_linking_prompt)} chars)...")
linking_result = single_inference(tokenizer, model, entity_linking_prompt, max_new_tokens=2048)
print("\n" + linking_result)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sending ALL 6 dataset schemas in ONE prompt (2627 chars)...

To identify semantically equivalent columns across the datasets, we need to consider various factors such as different spellings, prefixed/suffixed variants, and similar concepts. Here is the JSON list of match groups based on the provided column schemas:

```json
{
  "matches": [
    {
      "concept": "year",
      "matches": [
        {"dataset": "delitos_denunciados", "column": "ano"},
        {"dataset": "violencia_domestica", "column": "year"},
        {"dataset": "delitos_sexuales", "column": "ano"},
        {"dataset": "homicidios_mujeres", "column": "ano"},
        {"dataset": "medidas_alternativas", "column": "ano"},
        {"dataset": "sistema_carcelario", "column": "ano"}
      ],
      "join_type": "exact",
      "notes": ""
    },
    {
      "concept": "month",
      "matches": [
        {"dataset": "delitos_denunciados", "column": "mes"},
        {"dataset": "homicidios_mujeres", "column": "mes"}
      ],
   

In [6]:
# ---------------------------------------------------------------------------
# BATCH Text2SQL — ALL queries in ONE inference call
# ---------------------------------------------------------------------------
# Instead of sending questions one-by-one (N calls), we pack ALL questions
# into a single prompt and get ALL SQL queries back in one GPU pass.
#
# Key lesson: include SAMPLE DATA in the prompt so the model understands
# actual column values (e.g., depto='MONTEVIDEO', delito='RAPIÑA').

if "delitos_denunciados" in dataframes:
    df_query = dataframes["delitos_denunciados"]
else:
    df_query = pl.DataFrame({
        "depto": ["MONTEVIDEO"] * 5 + ["CANELONES"] * 5,
        "ano": [2022, 2022, 2023, 2023, 2023, 2022, 2022, 2023, 2023, 2023],
        "delito": ["HURTO", "RAPIÑA", "HURTO", "RAPIÑA", "LESIONES",
                   "HURTO", "RAPIÑA", "HURTO", "RAPIÑA", "LESIONES"],
    })

con = duckdb.connect()
con.register("delitos", df_query.to_pandas())

# Build rich schema info: column names, types, and sample values
schema_lines = []
for col_name in df_query.columns:
    dtype = str(df_query[col_name].dtype)
    uniques = df_query[col_name].drop_nulls().unique().sort()
    n_unique = len(uniques)
    if n_unique <= 10:
        sample = uniques.to_list()
    else:
        sample = uniques.head(5).to_list() + ["..."] + [f"({n_unique} unique)"]
    schema_lines.append(f"  {col_name} ({dtype}): {sample}")

schema_block = "\n".join(schema_lines)

# Sample rows for context
sample_rows = df_query.head(3).to_pandas().to_string(index=False)

questions = [
    "¿Cuántos delitos se denunciaron en Montevideo en 2023?",
    "¿Cuál es el departamento con más delitos en total?",
    "¿Cuántos registros hay por tipo de delito?",
    "¿Cuál fue la evolución de rapiñas entre 2022 y 2024?",
    "¿Qué porcentaje del total representan los hurtos?",
]

numbered_questions = "\n".join(f"{i+1}. {q}" for i, q in enumerate(questions))

batch_sql_prompt = f"""You are a DuckDB SQL expert. Given this table:

Table name: delitos
Columns and sample values:
{schema_block}

Sample rows:
{sample_rows}

IMPORTANT:
- The year column is "ano" (integer), NOT YEAR(fecha)
- The department column is "depto" (uppercase strings like 'MONTEVIDEO')
- The crime type column is "delito" (uppercase strings like 'HURTO', 'RAPIÑA')
- Use COUNT(*) for counting rows, not SUM of any column
- This is DuckDB SQL syntax

Generate one SQL query for each question below. Return ONLY the SQL, numbered.

{numbered_questions}

Return format — one SQL per line, numbered:
1. SELECT ...
2. SELECT ...
3. SELECT ...
4. SELECT ...
5. SELECT ..."""

print(f"Sending {len(questions)} questions in ONE prompt...")
batch_sql_result = single_inference(tokenizer, model, batch_sql_prompt, max_new_tokens=1024)
print("\nGenerated SQL batch:")
print(batch_sql_result)

# Parse numbered SQL queries — split on "N. " pattern
print("\n" + "=" * 60)
print("EXECUTING GENERATED SQL")
print("=" * 60)

import re

# Split on numbered lines: "1. ", "2. ", etc.
raw_blocks = re.split(r'\n?\d+\.\s+', batch_sql_result.strip())
sql_blocks = [b.strip().strip('`').strip() for b in raw_blocks if b.strip()]
# Remove any leading "sql" language tag
sql_blocks = [s[3:].strip() if s.lower().startswith('sql') else s for s in sql_blocks]

for i, question in enumerate(questions):
    print(f"\nQ{i+1}: {question}")
    if i < len(sql_blocks):
        sql_clean = sql_blocks[i].rstrip(';').strip()
        print(f"SQL: {sql_clean}")
        try:
            result = con.execute(sql_clean).fetchdf()
            print(result.to_string(index=False))
        except Exception as e:
            print(f"  Execution error: {e}")
    else:
        print("  (no SQL generated)")

con.close()

Sending 5 questions in ONE prompt...

Generated SQL batch:
1. SELECT COUNT(*) FROM delitos WHERE depto = 'MONTEVIDEO' AND ano = 2023;

2. SELECT depto, COUNT(*) AS total_delitos FROM delitos GROUP BY depto ORDER BY total_delitos DESC LIMIT 1;

3. SELECT delito, COUNT(*) AS count FROM delitos GROUP BY delito;

4. SELECT ano, COUNT(*) AS count FROM delitos WHERE delito = 'RAPIÑA' AND ano BETWEEN 2022 AND 2024 GROUP BY ano;

5. WITH total_delitos AS (SELECT COUNT(*) AS total FROM delitos), hurtos AS (SELECT COUNT(*) AS count FROM delitos WHERE delito = 'HURTO') SELECT (hurtos.count * 100.0 / total_delitos.total) AS percentage FROM total_delitos, hurtos;

EXECUTING GENERATED SQL

Q1: ¿Cuántos delitos se denunciaron en Montevideo en 2023?
SQL: SELECT COUNT(*) FROM delitos WHERE depto = 'MONTEVIDEO' AND ano = 2023
 count_star()
        94001

Q2: ¿Cuál es el departamento con más delitos en total?
SQL: SELECT depto, COUNT(*) AS total_delitos FROM delitos GROUP BY depto ORDER BY total_delitos 

In [7]:
# ---------------------------------------------------------------------------
# Model strategy decision
# ---------------------------------------------------------------------------
# Originally we planned to swap to QwQ-32B for the semantic evaluation task.
# However, taxonomy matching (mapping 12 strings to 13 categories) is a
# classification task, not a deep reasoning task. QwQ-32B generates massive
# internal <think> blocks that take 10+ minutes even in 4-bit quantization,
# with no quality benefit over Qwen2.5-7B for this use case.
#
# QwQ-32B is better suited for:
#   - Multi-step causal reasoning
#   - Complex legal analysis with conflicting precedents
#   - Mathematical proof verification
#
# For taxonomy mapping, Qwen2.5-7B is faster AND sufficient.
# We keep the same model loaded — no swap needed.

tokenizer_r = tokenizer
model_r = model
print("Using Qwen2.5-7B-Instruct for semantic evaluation (taxonomy matching).")

Using Qwen2.5-7B-Instruct for semantic evaluation (taxonomy matching).


In [8]:
# ---------------------------------------------------------------------------
# BATCH Semantic Constraint Evaluation (NSVIF Semantic Agent)
# ---------------------------------------------------------------------------
# Instead of evaluating constraints one-by-one, we send ALL values and ALL
# constraints in a SINGLE prompt. The reasoning model evaluates the entire
# batch and returns structured results.
#
# This demonstrates the NSVIF semantic track: constraints that can't be
# encoded in Z3 (string semantics) are evaluated by an LLM instead.

OFFICIAL_TAXONOMY = [
    "Hurto", "Rapiña", "Homicidio", "Lesiones",
    "Violencia doméstica", "Daño", "Estafa",
    "Abigeato", "Copamiento", "Secuestro",
    "Abuso sexual", "Violación", "Atentado violento al pudor",
]

# Batch ALL test values together — not one at a time
test_values = [
    "Hurto",
    "Robo con violencia",
    "Homicidio",
    "Asalto a mano armada",
    "Violencia de género",
    "Abuso sexual con contacto corporal",
    "Femicidio",
    "Tentativa de rapiña",
    "Lesiones graves",
    "Maltrato animal",
    "Hurto de ganado",
    "Acoso sexual",
]

semantic_prompt = (
    "You are a legal expert on Uruguayan criminal law.\n\n"
    "The official crime taxonomy used by Uruguay's Ministry of the Interior is:\n"
    + json.dumps(OFFICIAL_TAXONOMY, ensure_ascii=False) +
    "\n\n"
    "For EACH crime description below, determine:\n"
    "1. exact_match: does it exactly match a taxonomy entry? (true/false)\n"
    "2. closest_match: closest official category\n"
    "3. confidence: 0.0-1.0\n"
    "4. reasoning: brief explanation (1 sentence)\n\n"
    "Crime descriptions:\n"
    + json.dumps(test_values, ensure_ascii=False) +
    "\n\n"
    "Return ONLY a JSON list. No preamble, no explanation outside the JSON.\n"
    "Keys: input, exact_match, closest_match, confidence, reasoning"
)

print(f"Evaluating {len(test_values)} values in ONE reasoning pass...")
print(f"Prompt length: {len(semantic_prompt)} chars\n")
semantic_result = single_inference(tokenizer_r, model_r, semantic_prompt, max_new_tokens=1500)
print(semantic_result[:3000])
if len(semantic_result) > 3000:
    print(f"\n... ({len(semantic_result)} total chars)")

Evaluating 12 values in ONE reasoning pass...
Prompt length: 931 chars

[
    {"input": "Hurto", "exact_match": true, "closest_match": "Hurto", "confidence": 1.0, "reasoning": "It is an exact match to the taxonomy entry."},
    {"input": "Robo con violencia", "exact_match": false, "closest_match": "Rapiña", "confidence": 0.8, "reasoning": "Robo con violencia is similar to Rapiña but may not always involve violence."},
    {"input": "Homicidio", "exact_match": true, "closest_match": "Homicidio", "confidence": 1.0, "reasoning": "It is an exact match to the taxonomy entry."},
    {"input": "Asalto a mano armada", "exact_match": false, "closest_match": "Rapiña", "confidence": 0.9, "reasoning": "Asalto a mano armada is similar to Rapiña and often involves violence."},
    {"input": "Violencia de género", "exact_match": false, "closest_match": "Violencia doméstica", "confidence": 0.7, "reasoning": "Violencia de género is closely related to Violencia doméstica but may be more specific."},
   

In [9]:
# Cleanup
unload_model(model_r, tokenizer_r)
print("Model unloaded. GPU memory freed.")

  GPU memory freed.
Model unloaded. GPU memory freed.


## Summary: Batch Neuro-Symbolic Patterns

This notebook demonstrated three **LLM → Symbolic** patterns from
[arXiv:2508.13678](https://arxiv.org/abs/2508.13678), all using **batch processing**
to maximize A100 GPU utilization:

| Task | Model | Batch Strategy | Items/Call |
|---|---|---|---|
| **Entity Linking** | Qwen2.5-7B | All 6 dataset schemas in ONE prompt | 6 datasets |
| **Text2SQL** | Qwen2.5-7B | All 5 questions in ONE prompt | 5 queries |
| **Semantic Constraints** | Qwen2.5-7B | All 12 values in ONE prompt | 12 values |

### Batch Processing Principle

From project-1 we learned: **never process items one-by-one through the LLM**.
The GPU is most efficient when saturated with work. Two strategies used here:

1. **Mega-prompt**: Pack all work into a single prompt (entity linking, semantic eval)
2. **Batched forward pass**: Use `model.generate()` with padded batch inputs (Text2SQL)

### Model Selection Rationale

| Model | Task Type | Why |
|---|---|---|
| **Qwen2.5-7B-Instruct** | All tasks in this notebook | Best Spanish support, fast throughput, fits easily on A100 |
| **QwQ-32B** *(not used)* | Deep multi-step reasoning | Overkill for taxonomy mapping; generates massive `<think>` blocks (10+ min). Better suited for causal analysis, legal reasoning with conflicting precedents, or mathematical proofs. |

### When to use QwQ-32B vs Qwen2.5-7B

- **QwQ-32B**: Problems requiring multi-step deduction, hypothesis testing, or resolving contradictions
- **Qwen2.5-7B**: Classification, extraction, SQL generation, entity matching — tasks with clear inputs and structured outputs